# CMU-MOSI: Audio + Video attention-based adaptive fusion (text removed)

Same approach as the 3-modality notebook, with the text branch removed everywhere:

| Cell | What it does |
|---|---|
| 1 | Feature extraction (audio: MFCC+delta+ZCR+RMS, 28-d; video: 48x48 face crop, 2304-d). Writes `mosi_data.pkl` |
| 2 | `model.py`: BiLSTM encoders, audio<->video cross-attention, adaptive fusion gate (2 modalities) |
| 3 | Training and test evaluation |
| 4 | Ablation: adaptive gate vs fixed average fusion |

If `/kaggle/working/mosi_features/mosi_data.pkl` already exists from the earlier run, you can skip cell 1 (the text arrays in it are ignored).

Outputs are cleared: run the cells again, the old numbers belong to the text + audio + video model.

In [1]:
"""
ALL-IN-ONE script for CMU-MOSI AUDIO + VIDEO feature extraction on Kaggle (text removed).
Just run this single cell top to bottom -- it handles:
  1. Installing gdown (only thing needed beyond Kaggle's default packages)
  2. Downloading label.csv (skips if already present)
  3. Auto-finding your Raw/ dataset folder
  4. Extracting audio (librosa) and video (OpenCV Haar cascade) features
  5. Saving the final mosi_data.pkl  (keys: audio, vision, labels per split)

BEFORE RUNNING, MAKE SURE:
  - Your CMU-MOSI raw dataset is added as Input (mathurinache/cmu-mosi)
  - Internet is ON in notebook settings (needed for gdown + librosa/pandas if not cached)

NOTE: the transcript files are still used ONLY to list the clip ids, so that the
train/valid/test samples stay identical to the 3-modality experiment. No text is read or used.
"""

# ============================================================
# STEP 0: Install only what's truly needed (gdown for label download)
# ============================================================
import subprocess
subprocess.run(["pip", "install", "-q", "gdown"], check=True)

import os
import glob
import pickle
import numpy as np

# ============================================================
# STEP 1: Download label.csv if not already present
# ============================================================
LABEL_CSV_PATH = "/kaggle/working/label.csv"
LABEL_FILE_ID = "1dqyQI8iOHGjwofgq1DvAD-inW-zrTa2h"

if not os.path.exists(LABEL_CSV_PATH):
    print("label.csv not found -- downloading...")
    subprocess.run(["gdown", "--id", LABEL_FILE_ID, "-O", LABEL_CSV_PATH], check=True)
else:
    print("label.csv already present, skipping download.")

assert os.path.exists(LABEL_CSV_PATH), "label.csv download failed -- check the gdown output above for errors."


# ============================================================
# STEP 2: Auto-find the Raw/ dataset folder
# ============================================================
def find_raw_root(base="/kaggle/input"):
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > 4:
            dirs[:] = []
            continue
        if "Raw" in dirs:
            return os.path.join(root, "Raw")
    raise FileNotFoundError(
        f"Could not find a 'Raw' folder under {base}. "
        "Make sure the CMU-MOSI dataset is added as Input."
    )


RAW_ROOT = find_raw_root()
print("RAW_ROOT:", RAW_ROOT)

AUDIO_SEGMENTED_DIR = os.path.join(RAW_ROOT, "Audio", "WAV_16000", "Segmented")
TRANSCRIPT_DIR = os.path.join(RAW_ROOT, "Transcript", "Segmented")   # only used to list clip ids
VIDEO_DIR = os.path.join(RAW_ROOT, "Video", "Segmented")

OUTPUT_DIR = "/kaggle/working/mosi_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEQ_LEN_AUDIO = 50
SEQ_LEN_VIDEO = 50


# ============================================================
# STEP 3: Clip-id listing + sample matching (no text is read)
# ============================================================
def parse_clip_ids(path):
    ids = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or "_DELIM_" not in line:
                continue
            clip_id = line.split("_DELIM_", 1)[0]
            ids.append(clip_id.strip())
    return ids


def find_matched_samples():
    entries = []
    transcript_files = sorted(glob.glob(os.path.join(TRANSCRIPT_DIR, "*.annotprocessed")))
    print(f"Found {len(transcript_files)} videos (listed via transcript files)")
    for t_path in transcript_files:
        video_id = os.path.splitext(os.path.basename(t_path))[0]
        for clip_id in parse_clip_ids(t_path):
            wav_path = os.path.join(AUDIO_SEGMENTED_DIR, f"{video_id}_{clip_id}.wav")
            mp4_path = os.path.join(VIDEO_DIR, f"{video_id}_{clip_id}.mp4")
            if os.path.exists(wav_path) and os.path.exists(mp4_path):
                entries.append({
                    "video_id": video_id, "clip_id": clip_id,
                    "audio_path": wav_path, "video_path": mp4_path,
                })
    return entries


# ============================================================
# STEP 4: Audio features (librosa -- pre-installed on Kaggle)
# ============================================================
def extract_audio_features(wav_path, hop_length=512):
    import librosa
    y, sr = librosa.load(wav_path, sr=16000)
    if len(y) < hop_length:
        y = np.pad(y, (0, hop_length - len(y)))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length)
    delta = librosa.feature.delta(mfcc)
    zcr = librosa.feature.zero_crossing_rate(y=y, hop_length=hop_length)
    rms = librosa.feature.rms(y=y, hop_length=hop_length)
    feats = np.concatenate([mfcc, delta, zcr, rms], axis=0)
    return feats.T.astype(np.float32)  # (T, 28)


# ============================================================
# STEP 5: Video features (OpenCV Haar cascade -- pre-installed, no downloads)
# ============================================================
_FACE_CASCADE = None

def get_face_cascade():
    global _FACE_CASCADE
    if _FACE_CASCADE is None:
        import cv2
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        _FACE_CASCADE = cv2.CascadeClassifier(cascade_path)
    return _FACE_CASCADE


def extract_video_features(video_path, sample_every_n_frames=3, crop_size=48):
    import cv2
    cascade = get_face_cascade()
    cap = cv2.VideoCapture(video_path)
    frame_feats = []
    frame_idx = 0
    flat_dim = crop_size * crop_size
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_every_n_frames == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
            if len(faces) > 0:
                x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
                face_crop = gray[y:y + h, x:x + w]
                face_resized = cv2.resize(face_crop, (crop_size, crop_size))
                frame_feats.append((face_resized.flatten().astype(np.float32)) / 255.0)
            else:
                frame_feats.append(np.zeros(flat_dim, dtype=np.float32))
        frame_idx += 1
    cap.release()
    if len(frame_feats) == 0:
        return np.zeros((1, flat_dim), dtype=np.float32)
    return np.stack(frame_feats, axis=0)


def pad_or_truncate(seq, target_len):
    T, D = seq.shape
    if T >= target_len:
        return seq[:target_len]
    pad = np.zeros((target_len - T, D), dtype=np.float32)
    return np.concatenate([seq, pad], axis=0)


# ============================================================
# STEP 6: Main extraction loop
# ============================================================
def main():
    import pandas as pd

    labels_df = pd.read_csv(LABEL_CSV_PATH)
    labels_df["video_id"] = labels_df["video_id"].astype(str)
    labels_df["clip_id"] = labels_df["clip_id"].astype(str).str.lstrip("0").replace("", "0")
    labels_df["key"] = labels_df["video_id"] + "_" + labels_df["clip_id"]
    label_lookup = dict(zip(labels_df["key"], labels_df["label"]))
    mode_lookup = dict(zip(labels_df["key"], labels_df["mode"]))

    samples = find_matched_samples()
    print(f"Found {len(samples)} samples with matching audio + video")

    splits = {"train": {"audio": [], "vision": [], "labels": []},
              "valid": {"audio": [], "vision": [], "labels": []},
              "test":  {"audio": [], "vision": [], "labels": []}}

    matched_labels = 0
    for i, s in enumerate(samples):
        key = f"{s['video_id']}_{s['clip_id']}"
        if key not in label_lookup:
            continue
        matched_labels += 1
        split = mode_lookup[key]
        if split not in splits:
            continue
        try:
            audio_feat = extract_audio_features(s["audio_path"])
            video_feat = extract_video_features(s["video_path"])

            audio_feat = pad_or_truncate(audio_feat, SEQ_LEN_AUDIO)
            video_feat = pad_or_truncate(video_feat, SEQ_LEN_VIDEO)

            splits[split]["audio"].append(audio_feat)
            splits[split]["vision"].append(video_feat)
            splits[split]["labels"].append([label_lookup[key]])
        except Exception as e:
            print(f"Skipping {key}: {e}")

        if i % 100 == 0:
            print(f"Processed {i}/{len(samples)}")

    print(f"\n{matched_labels}/{len(samples)} samples had a matching label")

    final = {}
    for split in ["train", "valid", "test"]:
        final[split] = {
            "audio": np.stack(splits[split]["audio"]) if splits[split]["audio"] else np.zeros((0, SEQ_LEN_AUDIO, 28)),
            "vision": np.stack(splits[split]["vision"]) if splits[split]["vision"] else np.zeros((0, SEQ_LEN_VIDEO, 48 * 48)),
            "labels": np.array(splits[split]["labels"], dtype=np.float32),
        }
        print(f"{split}: {final[split]['labels'].shape[0]} samples, "
              f"audio_dim={final[split]['audio'].shape[-1]}, video_dim={final[split]['vision'].shape[-1]}")

    out_path = os.path.join(OUTPUT_DIR, "mosi_data.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(final, f)
    print(f"\nSaved -> {out_path}")


main()


label.csv not found -- downloading...


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1dqyQI8iOHGjwofgq1DvAD-inW-zrTa2h
To: /kaggle/working/label.csv
100%|██████████| 220k/220k [00:00<00:00, 9.32MB/s]


RAW_ROOT: /kaggle/input/datasets/mathurinache/cmu-mosi/Raw
Found 93 videos (listed via transcript files)
Found 2199 samples with matching audio + video
Processed 0/2199
Processed 100/2199
Processed 200/2199
Processed 300/2199
Processed 400/2199
Skipping 9qR7uwkblbs_33: when mode='interp', width=9 cannot exceed data.shape[axis]=7
Processed 500/2199
Processed 600/2199
Processed 700/2199
Processed 800/2199
Processed 900/2199
Processed 1000/2199
Processed 1100/2199
Processed 1200/2199
Processed 1300/2199
Processed 1400/2199
Processed 1500/2199
Processed 1600/2199
Processed 1700/2199
Processed 1800/2199
Processed 1900/2199
Processed 2000/2199
Processed 2100/2199

2199/2199 samples had a matching label
train: 1283 samples, audio_dim=28, video_dim=2304
valid: 229 samples, audio_dim=28, video_dim=2304
test: 686 samples, audio_dim=28, video_dim=2304

Saved -> /kaggle/working/mosi_features/mosi_data.pkl


In [2]:
%%writefile model.py
"""
Attention-Based Fusion with Adaptive Learning for CMU-MOSI  (AUDIO + VIDEO only)
=============================================================
Multimodal sentiment regression (audio + video) using:
  1. Per-modality encoders (BiLSTM/Linear projections)
  2. Cross-modal multi-head attention (audio attends video, video attends audio)
  3. Adaptive fusion gate -- a small network that LEARNS, per sample,
     how much weight to give each modality's fused representation
     before combining them into the final prediction.

Feature dims in this project (from the extraction cell):
  - audio (librosa MFCC + delta + ZCR + RMS) : 28
  - video (48x48 face crop, flattened)        : 2304
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------
# 1. Per-modality encoder: projects raw features into a shared d_model space
# ----------------------------------------------------------------------
class ModalityEncoder(nn.Module):
    """BiLSTM encoder + linear projection into common embedding space."""

    def __init__(self, input_dim, d_model, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.proj = nn.Linear(hidden_dim * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths=None):
        # x: (B, T, input_dim)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            out, _ = self.lstm(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        else:
            out, _ = self.lstm(x)  # (B, T, 2*hidden_dim)
        out = self.proj(out)  # (B, T, d_model)
        out = self.norm(out)
        return self.dropout(out)


# ----------------------------------------------------------------------
# 2. Cross-modal attention block: modality A (query) attends to modality B (key/value)
# ----------------------------------------------------------------------
class CrossModalAttention(nn.Module):
    """Standard multi-head cross attention: query modality attends over context modality."""

    def __init__(self, d_model, num_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, query, context, key_padding_mask=None):
        attn_out, attn_weights = self.mha(
            query, context, context, key_padding_mask=key_padding_mask
        )
        x = self.norm(query + attn_out)          # residual + norm
        x = self.norm2(x + self.ffn(x))           # feed-forward + residual + norm
        return x, attn_weights


# ----------------------------------------------------------------------
# 3. Adaptive Fusion Gate -- learns per-sample modality importance weights
# ----------------------------------------------------------------------
class AdaptiveFusionGate(nn.Module):
    """
    Takes pooled representations from each modality stream and learns
    scalar gate weights (summing to 1 via softmax) that adapt PER SAMPLE
    based on the content of that sample.
    """

    def __init__(self, d_model, num_modalities=2, hidden_dim=64):
        super().__init__()
        self.num_modalities = num_modalities
        self.gate_net = nn.Sequential(
            nn.Linear(d_model * num_modalities, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_modalities),
        )

    def forward(self, modality_reprs):
        # modality_reprs: list of (B, d_model) tensors, one per modality
        concat = torch.cat(modality_reprs, dim=-1)          # (B, d_model * M)
        gate_logits = self.gate_net(concat)                  # (B, M)
        gate_weights = F.softmax(gate_logits, dim=-1)         # (B, M) sums to 1

        stacked = torch.stack(modality_reprs, dim=1)          # (B, M, d_model)
        fused = torch.sum(gate_weights.unsqueeze(-1) * stacked, dim=1)  # (B, d_model)
        return fused, gate_weights


# ----------------------------------------------------------------------
# 4. Full Model: Attention-Based Fusion with Adaptive Learning (audio + video)
# ----------------------------------------------------------------------
class AttentiveAdaptiveFusionModel(nn.Module):
    def __init__(
        self,
        audio_dim=28,
        video_dim=2304,
        d_model=128,
        num_heads=4,
        dropout=0.2,
    ):
        super().__init__()

        # --- per-modality encoders ---
        self.audio_encoder = ModalityEncoder(audio_dim, d_model, dropout=dropout)
        self.video_encoder = ModalityEncoder(video_dim, d_model, dropout=dropout)

        # --- cross-modal attention: audio <-> video ---
        self.audio_attends_video = CrossModalAttention(d_model, num_heads, dropout)
        self.video_attends_audio = CrossModalAttention(d_model, num_heads, dropout)

        # --- adaptive fusion gate (2 modalities) ---
        self.fusion_gate = AdaptiveFusionGate(d_model, num_modalities=2)

        # --- prediction head ---
        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),   # CMU-MOSI: scalar sentiment score in [-3, 3]
        )

    @staticmethod
    def _masked_mean_pool(x, mask=None):
        # x: (B, T, d_model), mask: (B, T) True where PAD
        if mask is None:
            return x.mean(dim=1)
        lengths = (~mask).sum(dim=1, keepdim=True).clamp(min=1)
        x = x.masked_fill(mask.unsqueeze(-1), 0.0)
        return x.sum(dim=1) / lengths

    def forward(self, audio, video, audio_mask=None, video_mask=None):
        """
        audio: (B, T_a, audio_dim)
        video: (B, T_v, video_dim)
        *_mask: (B, T) boolean, True at PAD positions (optional)
        """
        # 1) encode each modality into shared space
        a = self.audio_encoder(audio)  # (B, T_a, d_model)
        v = self.video_encoder(video)  # (B, T_v, d_model)

        # 2) cross-modal attention (each modality queries the other)
        a_fused, _ = self.audio_attends_video(a, v, key_padding_mask=video_mask)
        v_fused, _ = self.video_attends_audio(v, a, key_padding_mask=audio_mask)

        # 3) pool each modality's sequence into a single vector
        a_pooled = self._masked_mean_pool(a_fused, audio_mask)
        v_pooled = self._masked_mean_pool(v_fused, video_mask)

        # 4) adaptive fusion -- learns per-sample modality weighting
        fused_repr, gate_weights = self.fusion_gate([a_pooled, v_pooled])

        # 5) prediction
        pred = self.regressor(fused_repr).squeeze(-1)  # (B,)

        return pred, gate_weights  # return gate_weights too, useful for analysis/interpretability


# ----------------------------------------------------------------------
# Quick shape sanity check (run: python model.py)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    B = 8
    T_a, T_v = 50, 50
    AUDIO_DIM, VIDEO_DIM = 28, 2304

    model = AttentiveAdaptiveFusionModel(
        audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM, d_model=128
    )

    audio = torch.randn(B, T_a, AUDIO_DIM)
    video = torch.randn(B, T_v, VIDEO_DIM)

    pred, gates = model(audio, video)
    print("Prediction shape:", pred.shape)          # (B,)
    print("Gate weights shape:", gates.shape)        # (B, 2)
    print("Sample gate weights (audio, video):\n", gates[0])
    print("Total trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Writing model.py


In [3]:
"""
Training script for Attention-Based Fusion with Adaptive Learning on CMU-MOSI
(AUDIO + VIDEO only).

Expects the pickle written by the extraction cell:

    mosi_data.pkl -> {
        'train': {'audio': (N,T,28), 'vision': (N,T,2304), 'labels': (N,1)},
        'valid': {...},
        'test':  {...},
    }

(If your pickle still contains a 'text' key from the older 3-modality run, it is simply ignored.)
"""

import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score

from model import AttentiveAdaptiveFusionModel

# ---------------- CONFIG ----------------
DATA_PATH = "/kaggle/working/mosi_features/mosi_data.pkl"     # <-- your extracted features
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEY_MAP = {"audio": "audio", "video": "vision", "labels": "labels"}


class MOSIDataset(Dataset):
    def __init__(self, split_dict):
        self.audio = split_dict[KEY_MAP["audio"]].astype(np.float32)
        self.video = split_dict[KEY_MAP["video"]].astype(np.float32)
        labels = split_dict[KEY_MAP["labels"]]
        self.labels = labels.astype(np.float32).reshape(-1)

        # replace any inf/nan in audio/video
        self.audio = np.nan_to_num(self.audio, nan=0.0, posinf=0.0, neginf=0.0)
        self.video = np.nan_to_num(self.video, nan=0.0, posinf=0.0, neginf=0.0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.audio[idx]),
            torch.tensor(self.video[idx]),
            torch.tensor(self.labels[idx]),
        )


def load_data(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["train"], data["valid"], data["test"]


def compute_metrics(preds, labels):
    preds = np.array(preds)
    labels = np.array(labels)
    mae = mean_absolute_error(labels, preds)
    corr = np.corrcoef(preds, labels)[0, 1]
    # binary sentiment accuracy (positive/negative), standard MOSI eval protocol
    bin_preds = (preds >= 0).astype(int)
    bin_labels = (labels >= 0).astype(int)
    acc2 = accuracy_score(bin_labels, bin_preds)
    f1 = f1_score(bin_labels, bin_preds, average="weighted")
    return {"MAE": mae, "Corr": corr, "Acc2": acc2, "F1": f1}


def run_epoch(model, loader, optimizer=None, criterion=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for audio, video, labels in loader:
            audio, video, labels = audio.to(DEVICE), video.to(DEVICE), labels.to(DEVICE)
            preds, gate_weights = model(audio, video)
            loss = criterion(preds, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(all_preds, all_labels)
    return avg_loss, metrics


def main():
    train_d, valid_d, test_d = load_data(DATA_PATH)

    train_ds = MOSIDataset(train_d)
    valid_ds = MOSIDataset(valid_d)
    test_ds = MOSIDataset(test_d)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    audio_dim = train_ds.audio.shape[-1]
    video_dim = train_ds.video.shape[-1]
    print(f"Feature dims -> audio: {audio_dim}, video: {video_dim}")

    model = AttentiveAdaptiveFusionModel(
        audio_dim=audio_dim, video_dim=video_dim, d_model=D_MODEL
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=4
    )
    criterion = nn.L1Loss()  # MAE loss, standard for MOSI regression

    best_val_mae = float("inf")
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_metrics = run_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics = run_epoch(model, valid_loader, None, criterion)
        scheduler.step(val_loss)

        print(
            f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | "
            f"val_MAE {val_metrics['MAE']:.4f} | val_Acc2 {val_metrics['Acc2']:.4f} | "
            f"val_F1 {val_metrics['F1']:.4f} | val_Corr {val_metrics['Corr']:.4f}"
        )

        if val_metrics["MAE"] < best_val_mae:
            best_val_mae = val_metrics["MAE"]
            torch.save(model.state_dict(), "best_model.pt")

    # final test evaluation with best checkpoint
    model.load_state_dict(torch.load("best_model.pt"))
    test_loss, test_metrics = run_epoch(model, test_loader, None, criterion)
    print("\n=== TEST RESULTS ===")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")


if __name__ == "__main__":
    main()


Feature dims -> audio: 28, video: 2304
Epoch 01 | train_loss 1.3070 | val_MAE 1.4061 | val_Acc2 0.6026 | val_F1 0.4577 | val_Corr 0.0660
Epoch 02 | train_loss 1.2685 | val_MAE 1.4106 | val_Acc2 0.5808 | val_F1 0.5662 | val_Corr 0.0558
Epoch 03 | train_loss 1.2335 | val_MAE 1.4263 | val_Acc2 0.5808 | val_F1 0.5606 | val_Corr 0.0426
Epoch 04 | train_loss 1.1923 | val_MAE 1.4621 | val_Acc2 0.5852 | val_F1 0.4562 | val_Corr 0.0411
Epoch 05 | train_loss 1.1707 | val_MAE 1.4481 | val_Acc2 0.5590 | val_F1 0.5156 | val_Corr 0.0074
Epoch 06 | train_loss 1.1374 | val_MAE 1.4108 | val_Acc2 0.5895 | val_F1 0.5506 | val_Corr 0.0979
Epoch 07 | train_loss 1.0950 | val_MAE 1.4827 | val_Acc2 0.5371 | val_F1 0.5293 | val_Corr -0.0187
Epoch 08 | train_loss 1.0786 | val_MAE 1.4038 | val_Acc2 0.5939 | val_F1 0.5446 | val_Corr 0.1817
Epoch 09 | train_loss 1.0480 | val_MAE 1.4189 | val_Acc2 0.5808 | val_F1 0.5808 | val_Corr 0.1087
Epoch 10 | train_loss 1.0348 | val_MAE 1.4155 | val_Acc2 0.5939 | val_F1 0.591

In [4]:
"""
Ablation Study: Adaptive Fusion Gate vs Fixed (Average) Fusion  (AUDIO + VIDEO only)
==================================================================
Trains TWO versions of the fusion model on the SAME CMU-MOSI data/splits:

  1. AdaptiveFusionModel -- cross-modal attention + per-sample ADAPTIVE gate
     that learns modality weights (audio, video)
  2. FixedFusionModel    -- IDENTICAL architecture (same encoders, same
     cross-modal attention) EXCEPT the adaptive gate is replaced with a
     simple fixed average of the two pooled modality representations.

Comparing their test metrics isolates the contribution of the adaptive gate.
Also prints the ADAPTIVE model's average gate weights (audio, video).
(A gate weight shows what the gate used, not which modality is more informative.)

Run this AFTER the extraction cell has created mosi_data.pkl.
"""

import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score

# ---------------- CONFIG ----------------
DATA_PATH = "/kaggle/working/mosi_features/mosi_data.pkl"
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEY_MAP = {"audio": "audio", "video": "vision", "labels": "labels"}


# ----------------------------------------------------------------------
# Shared building blocks (same as model.py)
# ----------------------------------------------------------------------
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, d_model, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.proj = nn.Linear(hidden_dim * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.proj(out)
        out = self.norm(out)
        return self.dropout(out)


class CrossModalAttention(nn.Module):
    def __init__(self, d_model, num_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, query, context):
        attn_out, _ = self.mha(query, context, context)
        x = self.norm(query + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x


def masked_mean_pool(x):
    return x.mean(dim=1)


class BackboneFusionModel(nn.Module):
    """Shared trunk: encoders + cross-modal attention. Subclasses differ only in fusion step."""

    def __init__(self, audio_dim, video_dim, d_model=128, num_heads=4, dropout=0.2):
        super().__init__()
        self.audio_encoder = ModalityEncoder(audio_dim, d_model, dropout=dropout)
        self.video_encoder = ModalityEncoder(video_dim, d_model, dropout=dropout)

        self.audio_attends_video = CrossModalAttention(d_model, num_heads, dropout)
        self.video_attends_audio = CrossModalAttention(d_model, num_heads, dropout)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1),
        )

    def encode_and_attend(self, audio, video):
        a = self.audio_encoder(audio)
        v = self.video_encoder(video)

        a_fused = self.audio_attends_video(a, v)
        v_fused = self.video_attends_audio(v, a)

        a_pooled = masked_mean_pool(a_fused)
        v_pooled = masked_mean_pool(v_fused)
        return a_pooled, v_pooled


class AdaptiveFusionGate(nn.Module):
    def __init__(self, d_model, num_modalities=2, hidden_dim=64):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(d_model * num_modalities, hidden_dim), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(hidden_dim, num_modalities),
        )

    def forward(self, modality_reprs):
        concat = torch.cat(modality_reprs, dim=-1)
        gate_weights = F.softmax(self.gate_net(concat), dim=-1)
        stacked = torch.stack(modality_reprs, dim=1)
        fused = torch.sum(gate_weights.unsqueeze(-1) * stacked, dim=1)
        return fused, gate_weights


class AdaptiveFusionModel(BackboneFusionModel):
    """FULL model: adaptive, per-sample gate weighting."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        d_model = kwargs.get("d_model", 128)
        self.fusion_gate = AdaptiveFusionGate(d_model, num_modalities=2)

    def forward(self, audio, video):
        a_p, v_p = self.encode_and_attend(audio, video)
        fused, gate_weights = self.fusion_gate([a_p, v_p])
        pred = self.regressor(fused).squeeze(-1)
        return pred, gate_weights


class FixedFusionModel(BackboneFusionModel):
    """ABLATION baseline: identical trunk, but fusion = simple FIXED average (no adaptivity)."""

    def forward(self, audio, video):
        a_p, v_p = self.encode_and_attend(audio, video)
        fused = (a_p + v_p) / 2.0  # fixed, equal weighting -- no learned gate
        pred = self.regressor(fused).squeeze(-1)
        fixed_weights = torch.full((audio.size(0), 2), 1.0 / 2, device=audio.device)
        return pred, fixed_weights


# ----------------------------------------------------------------------
# Dataset
# ----------------------------------------------------------------------
class MOSIDataset(Dataset):
    def __init__(self, split_dict):
        self.audio = np.nan_to_num(split_dict[KEY_MAP["audio"]].astype(np.float32))
        self.video = np.nan_to_num(split_dict[KEY_MAP["video"]].astype(np.float32))
        self.labels = split_dict[KEY_MAP["labels"]].astype(np.float32).reshape(-1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.audio[idx]), torch.tensor(self.video[idx]),
                torch.tensor(self.labels[idx]))


def compute_metrics(preds, labels):
    preds, labels = np.array(preds), np.array(labels)
    mae = mean_absolute_error(labels, preds)
    corr = np.corrcoef(preds, labels)[0, 1]
    bin_preds = (preds >= 0).astype(int)
    bin_labels = (labels >= 0).astype(int)
    return {"MAE": mae, "Corr": corr, "Acc2": accuracy_score(bin_labels, bin_preds),
            "F1": f1_score(bin_labels, bin_preds, average="weighted")}


def run_epoch(model, loader, optimizer=None, criterion=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels, all_gates = 0.0, [], [], []
    with torch.set_grad_enabled(is_train):
        for audio, video, labels in loader:
            audio, video, labels = (t.to(DEVICE) for t in (audio, video, labels))
            preds, gate_weights = model(audio, video)
            loss = criterion(preds, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_gates.append(gate_weights.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(all_preds, all_labels)
    avg_gates = np.concatenate(all_gates, axis=0).mean(axis=0)
    return avg_loss, metrics, avg_gates


def train_model(model, train_loader, valid_loader, test_loader, name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
    criterion = nn.L1Loss()

    best_val_mae = float("inf")
    for epoch in range(1, EPOCHS + 1):
        train_loss, _, _ = run_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics, _ = run_epoch(model, valid_loader, None, criterion)
        scheduler.step(val_loss)
        if val_metrics["MAE"] < best_val_mae:
            best_val_mae = val_metrics["MAE"]
            torch.save(model.state_dict(), f"best_{name}.pt")
        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"[{name}] Epoch {epoch:02d} | val_MAE {val_metrics['MAE']:.4f} | "
                  f"val_Acc2 {val_metrics['Acc2']:.4f}")

    model.load_state_dict(torch.load(f"best_{name}.pt"))
    test_loss, test_metrics, test_gates = run_epoch(model, test_loader, None, criterion)
    return test_metrics, test_gates


def main():
    with open(DATA_PATH, "rb") as f:
        data = pickle.load(f)

    train_ds = MOSIDataset(data["train"])
    valid_ds = MOSIDataset(data["valid"])
    test_ds = MOSIDataset(data["test"])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    audio_dim = train_ds.audio.shape[-1]
    video_dim = train_ds.video.shape[-1]
    print(f"Dims -> audio: {audio_dim}, video: {video_dim}\n")

    print("=" * 60)
    print("Training FULL model (Adaptive Gate)...")
    print("=" * 60)
    adaptive_model = AdaptiveFusionModel(audio_dim, video_dim, d_model=D_MODEL).to(DEVICE)
    adaptive_metrics, adaptive_gates = train_model(
        adaptive_model, train_loader, valid_loader, test_loader, "adaptive")

    print("\n" + "=" * 60)
    print("Training ABLATION baseline (Fixed/Average Fusion, no adaptive gate)...")
    print("=" * 60)
    fixed_model = FixedFusionModel(audio_dim, video_dim, d_model=D_MODEL).to(DEVICE)
    fixed_metrics, _ = train_model(
        fixed_model, train_loader, valid_loader, test_loader, "fixed")

    print("\n" + "=" * 60)
    print("ABLATION RESULTS (same data, same encoders/attention, only fusion step differs)")
    print("=" * 60)
    print(f"{'Metric':<10} {'Adaptive Gate':<15} {'Fixed (Avg) Fusion':<20} {'Improvement'}")
    for m in ["MAE", "Corr", "Acc2", "F1"]:
        adapt_v, fixed_v = adaptive_metrics[m], fixed_metrics[m]
        better = "lower is better" if m in ("MAE",) else "higher is better"
        diff = (fixed_v - adapt_v) if m == "MAE" else (adapt_v - fixed_v)
        print(f"{m:<10} {adapt_v:<15.4f} {fixed_v:<20.4f} {diff:+.4f} ({better})")

    print("\nAdaptive gate average weights on TEST set:")
    print(f"  Audio: {adaptive_gates[0]:.4f}")
    print(f"  Video: {adaptive_gates[1]:.4f}")
    print("\n(Gate weights show what the gate used, not which modality is more informative.)")


if __name__ == "__main__":
    main()


Dims -> audio: 28, video: 2304

Training FULL model (Adaptive Gate)...
[adaptive] Epoch 10 | val_MAE 1.3236 | val_Acc2 0.6463
[adaptive] Epoch 20 | val_MAE 1.3438 | val_Acc2 0.6419
[adaptive] Epoch 30 | val_MAE 1.3778 | val_Acc2 0.5939
[adaptive] Epoch 40 | val_MAE 1.3702 | val_Acc2 0.6157

Training ABLATION baseline (Fixed/Average Fusion, no adaptive gate)...
[fixed] Epoch 10 | val_MAE 1.3575 | val_Acc2 0.6157
[fixed] Epoch 20 | val_MAE 1.4410 | val_Acc2 0.5939
[fixed] Epoch 30 | val_MAE 1.4907 | val_Acc2 0.5895
[fixed] Epoch 40 | val_MAE 1.4919 | val_Acc2 0.5983

ABLATION RESULTS (same data, same encoders/attention, only fusion step differs)
Metric     Adaptive Gate   Fixed (Avg) Fusion   Improvement
MAE        1.6996          1.7037               +0.0041 (lower is better)
Corr       -0.0888         -0.0500              -0.0388 (higher is better)
Acc2       0.4781          0.4548               +0.0233 (higher is better)
F1         0.4731          0.4324               +0.0407 (higher 